# Complex Structured Output with `cf_structured()`

PydanticAI's built-in structured output uses tool calling, which breaks on Workers AI for complex schemas. `cf_structured()` handles this by calling the API directly with schema injection + custom retry logic.

Tested on **all 6 major Workers AI models** with 7+ nested Pydantic models.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

## Define a complex schema

4 nested models, `Literal` types, `list[NestedModel]` — the kind of schema you'd use in production.

In [2]:
from typing import Literal
from pydantic import BaseModel
from pydantic_ai_cloudflare import schema_stats

class NewsItem(BaseModel):
    headline: str
    summary: str

class TriggerEvent(BaseModel):
    event: str
    priority: Literal["HIGH", "MEDIUM", "LOW"]

class Company(BaseModel):
    name: str
    industry: str
    employees: str

class MarketIntel(BaseModel):
    news: list[NewsItem]
    triggers: list[TriggerEvent]

class NextStep(BaseModel):
    action: str
    priority: Literal["HIGH", "MEDIUM", "LOW"]

class Report(BaseModel):
    overview: str
    company: Company
    market: MarketIntel
    next_steps: list[NextStep]

stats = schema_stats(Report)
print(f"Schema: {stats['total_chars']} chars, {stats['nested_model_count']} nested models")
print(f"Simplified: {stats['simplified_chars']} chars ({stats['reduction']} reduction)")
print(f"Recommendation: {stats['recommendation']}")

Schema: 1793 chars, 5 nested models
Simplified: 1372 chars (23% reduction)
Recommendation: Good -- should work reliably with any Workers AI model.


## Run on Llama 3.3 70B

In [3]:
from pydantic_ai_cloudflare import cf_structured_sync

result = cf_structured_sync(
    "Report on 'DataFlow', a fictional IoT analytics company. "
    "300 employees. 2 news, 2 triggers, 3 next steps.",
    Report,
    model="@cf/meta/llama-3.3-70b-instruct-fp8-fast",
)

print(f"Company: {result.company.name} ({result.company.industry})")
print(f"Employees: {result.company.employees}")
print(f"\nNews ({len(result.market.news)}):")
for n in result.market.news:
    print(f"  - {n.headline}")
print(f"\nTriggers ({len(result.market.triggers)}):")
for t in result.market.triggers:
    print(f"  [{t.priority}] {t.event}")
print(f"\nNext steps ({len(result.next_steps)}):")
for s in result.next_steps:
    print(f"  [{s.priority}] {s.action}")

Company: DataFlow (IoT Analytics)
Employees: 300

News (2):
  - DataFlow Raises $50M Series B
  - DataFlow Launches Real-Time Dashboard

Triggers (2):
  [HIGH] Growing demand for IoT analytics
  [MEDIUM] New competitor entering the market

Next steps (3):
  [HIGH] Expand sales team
  [MEDIUM] Launch enterprise tier
  [LOW] Explore international markets


## Test across all models

Works on every major Workers AI model:

In [4]:
import asyncio, time
from pydantic_ai_cloudflare import cf_structured

prompt = (
    "Report on 'DataFlow', a fictional IoT analytics company. "
    "300 employees. 2 news, 2 triggers, 3 next steps."
)

models = [
    ("Llama 3.3 70B", "@cf/meta/llama-3.3-70b-instruct-fp8-fast"),
    ("Qwen 3 30B", "@cf/qwen/qwen3-30b-a3b-fp8"),
    ("Kimi K2.6", "@cf/moonshotai/kimi-k2.6"),
    ("GLM 4.7 Flash", "@cf/zai-org/glm-4.7-flash"),
]

async def test_all():
    print("cf_structured() across models:")
    for name, mid in models:
        start = time.time()
        try:
            r = await cf_structured(prompt, Report, model=mid)
            t = time.time() - start
            print(f"  {name:20s} \u2713 {t:5.1f}s | {r.company.name}, {len(r.market.news)} news, {len(r.next_steps)} steps")
        except Exception as e:
            t = time.time() - start
            print(f"  {name:20s} \u2717 {t:5.1f}s | {e}")

await test_all()

cf_structured() across models:
  Llama 3.3 70B        ✓ 13.7s | DataFlow, 2 news, 3 steps
  Qwen 3 30B           ✓  8.7s | DataFlow, 2 news, 3 steps
  Kimi K2.6            ✓ 19.3s | DataFlow, 2 news, 3 steps
  GLM 4.7 Flash        ✓  8.4s | DataFlow, 2 news, 3 steps


## How `cf_structured()` works

1. Takes your Pydantic model → generates JSON schema
2. Simplifies the schema (strips descriptions/defaults, ~25% smaller)
3. Injects schema into system prompt with strict formatting rules
4. Sets `response_format: json_object` to force valid JSON
5. Parses response (handles dict content, markdown fences, prose)
6. Validates with Pydantic `model_validate()`
7. On failure: retries with the validation error as feedback

This is the same approach `langchain-cloudflare` uses internally for `with_structured_output(method='json_schema')`, adapted for PydanticAI.